In [ ]:
import os


In [ ]:
import numpy as np


In [ ]:
import pandas as pd


In [ ]:
import matplotlib


In [ ]:
matplotlib.use("Agg")


In [ ]:
import matplotlib.pyplot as plt


In [ ]:
np.random.seed(123)


In [ ]:
os.makedirs("data_raw", exist_ok=True)


In [ ]:
os.makedirs("data_processed", exist_ok=True)


In [ ]:
os.makedirs("figures", exist_ok=True)


In [ ]:
os.makedirs("outputs", exist_ok=True)


In [ ]:
os.makedirs("src", exist_ok=True)


In [ ]:
n = 5000


In [ ]:
x_true = np.random.normal(loc=0.0, scale=1.0, size=n)


In [ ]:
logit_p = 1.0 * x_true


In [ ]:
p = 1.0 / (1.0 + np.exp(-logit_p))


In [ ]:
d = np.random.binomial(n=1, p=p, size=n)


In [ ]:
os.makedirs("09_data_quality/demo/data_raw", exist_ok=True)
os.makedirs("09_data_quality/demo/data_processed", exist_ok=True)
os.makedirs("09_data_quality/demo/figures", exist_ok=True)
os.makedirs("09_data_quality/demo/outputs", exist_ok=True)
os.makedirs("09_data_quality/demo/src", exist_ok=True)


In [ ]:
tau = 1.0


In [ ]:
beta = 1.0


In [ ]:
eps_y = np.random.normal(loc=0.0, scale=1.0, size=n)


In [ ]:
y = tau * d + beta * x_true + eps_y


In [ ]:
eps_pl = np.random.normal(loc=0.0, scale=1.0, size=n)


In [ ]:
y_placebo = 0.0 * d + beta * x_true + eps_pl


In [ ]:
df_base = pd.DataFrame(
    {
        "y": y,
        "y_placebo": y_placebo,
        "d": d,
        "x_true": x_true,
    }
)



In [ ]:
print("\n--- Data preview ---")



--- Data preview ---


In [ ]:
print(df_base.head())


          y  y_placebo  d    x_true
0  1.083028  -0.003808  1 -1.085631
1  2.334532   2.009014  1  0.997345
2  0.478507   0.826371  0  0.282978
3 -1.939787  -2.426575  0 -1.506295
4  0.069274  -0.754683  1 -0.578600


In [ ]:
print("\nTreatment rate:", round(df_base["d"].mean(), 4))



Treatment rate: 0.504


In [ ]:
sigma_u_grid = [0.0, 0.2, 0.5, 1.0, 2.0]
R = 30  # repetitions (to see variability from measurement error draws)


In [ ]:
R = 30


In [ ]:
validation_share = 0.20


In [ ]:
validation_idx = np.random.choice(np.arange(n), size=int(validation_share * n), replace=False)


In [ ]:
is_validation = np.zeros(n, dtype=bool)


In [ ]:
is_validation[validation_idx] = True


In [ ]:
rows = []


In [ ]:
print("\n--- Running measurement error simulations ---")



--- Running measurement error simulations ---


In [ ]:
for sigma_u in sigma_u_grid:
    tau_oracle_list = []
    tau_naive_list = []
    tau_cal_list = []
    tau_placebo_list = []
    beta_oracle_list = []
    beta_naive_list = []
    beta_cal_list = []
    for r in range(R):
        # Draw measurement error and observed covariate
        u = np.random.normal(loc=0.0, scale=sigma_u, size=n)
        x_obs = x_true + u
        # Build design matrices with intercept column
        ones = np.ones(n)
        # (A) Oracle regression: y ~ 1 + d + x_true
        X_oracle = np.column_stack([ones, d, x_true])
        coef_oracle, _, _, _ = np.linalg.lstsq(X_oracle, y, rcond=None)
        # coef_oracle: [intercept, d, x_true]
        tau_oracle_list.append(coef_oracle[1])
        beta_oracle_list.append(coef_oracle[2])
        # (B) Naive regression: y ~ 1 + d + x_obs
        X_naive = np.column_stack([ones, d, x_obs])
        coef_naive, _, _, _ = np.linalg.lstsq(X_naive, y, rcond=None)
        tau_naive_list.append(coef_naive[1])
        beta_naive_list.append(coef_naive[2])
        # (C) Regression calibration (validation subsample):
        #     estimate x_true ~ x_obs on validation sample, predict x_hat for all.
        ones_val = np.ones(int(validation_share * n))
        X_cal_val = np.column_stack([ones_val, x_obs[is_validation]])
        coef_cal, _, _, _ = np.linalg.lstsq(X_cal_val, x_true[is_validation], rcond=None)
        x_hat = coef_cal[0] + coef_cal[1] * x_obs
        X_calibrated = np.column_stack([ones, d, x_hat])
        coef_calibrated, _, _, _ = np.linalg.lstsq(X_calibrated, y, rcond=None)
        tau_cal_list.append(coef_calibrated[1])
        beta_cal_list.append(coef_calibrated[2])
        # Outcome placebo: y_placebo ~ 1 + d + x_obs
        X_placebo = np.column_stack([ones, d, x_obs])
        coef_placebo, _, _, _ = np.linalg.lstsq(X_placebo, y_placebo, rcond=None)
        tau_placebo_list.append(coef_placebo[1])
    # Summaries per sigma_u
    rows.append(
        {
            "sigma_u": sigma_u,
            "tau_true": tau,
            "tau_oracle_mean": float(np.mean(tau_oracle_list)),
            "tau_naive_mean": float(np.mean(tau_naive_list)),
            "tau_cal_mean": float(np.mean(tau_cal_list)),
            "tau_placebo_mean": float(np.mean(tau_placebo_list)),
            "tau_oracle_q025": float(np.quantile(tau_oracle_list, 0.025)),
            "tau_oracle_q975": float(np.quantile(tau_oracle_list, 0.975)),
            "tau_naive_q025": float(np.quantile(tau_naive_list, 0.025)),
            "tau_naive_q975": float(np.quantile(tau_naive_list, 0.975)),
            "tau_cal_q025": float(np.quantile(tau_cal_list, 0.025)),
            "tau_cal_q975": float(np.quantile(tau_cal_list, 0.975)),
            "beta_true": beta,
            "beta_oracle_mean": float(np.mean(beta_oracle_list)),
            "beta_naive_mean": float(np.mean(beta_naive_list)),
            "beta_cal_mean": float(np.mean(beta_cal_list)),
        }
    )
    print(f"  done sigma_u={sigma_u}")



  done sigma_u=0.0
  done sigma_u=0.2
  done sigma_u=0.5
  done sigma_u=1.0
  done sigma_u=2.0


In [ ]:
results = pd.DataFrame(rows)


In [ ]:
print("\n--- Summary (means) ---")



--- Summary (means) ---


In [ ]:
print(results[["sigma_u", "tau_true", "tau_oracle_mean", "tau_naive_mean", "tau_cal_mean", "tau_placebo_mean"]].to_string(index=False))


 sigma_u  tau_true  tau_oracle_mean  tau_naive_mean  tau_cal_mean  tau_placebo_mean
     0.0       1.0         0.973679        0.973679      0.973679         -0.003150
     0.2       1.0         0.973679        1.009563      1.009563          0.032990
     0.5       1.0         0.973679        1.175672      1.175672          0.198365
     1.0       1.0         0.973679        1.445083      1.445083          0.463515
     2.0       1.0         0.973679        1.679230      1.679230          0.695895


In [ ]:
results.to_csv("outputs/measurement_error_results.csv", index=False)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.4.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\core\generic.py", line 3988, in to_csv
    return DataFrameRenderer(formatter).to_csv(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\io\formats\format.py", line 1025, in to_csv
    csv_formatter.save()
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pandas\io\formats\csvs.py", li

In [ ]:
results.to_csv("09_data_quality/demo/outputs/measurement_error_results.csv", index=False)


In [ ]:
plt.figure(figsize=(8, 5))


Figure(800x500)


In [ ]:
plt.plot(results["sigma_u"], results["tau_oracle_mean"], marker="o", label="Oracle: y ~ d + x_true")


In [ ]:
plt.plot(results["sigma_u"], results["tau_naive_mean"], marker="o", label="Naive: y ~ d + x_obs")


In [ ]:
plt.plot(results["sigma_u"], results["tau_cal_mean"], marker="o", label="Calibration: y ~ d + x_hat")


In [ ]:
plt.plot(results["sigma_u"], results["tau_placebo_mean"], marker="o", label="Outcome placebo: y_pl ~ d + x_obs")


In [ ]:
plt.plot(results["sigma_u"], results["tau_cal_mean"], marker="o", label="Calibration: y ~ d + x_hat")


In [ ]:
plt.plot(results["sigma_u"], results["tau_placebo_mean"], marker="o", label="Outcome placebo: y_pl ~ d + x_obs")


In [ ]:
plt.axhline(tau, linestyle="--", label="True tau")


Line2D(True tau)


In [ ]:
plt.title("Estimated treatment effect vs measurement error in confounder")


Text(0.5, 1.0, 'Estimated treatment effect vs measurement error in confounder')


In [ ]:
plt.xlabel("Measurement error SD (sigma_u)")


Text(0.5, 0, 'Measurement error SD (sigma_u)')


In [ ]:
plt.ylabel("Estimated coefficient on d")


Text(0, 0.5, 'Estimated coefficient on d')


In [ ]:
plt.legend()


Legend


In [ ]:
plt.tight_layout()


In [ ]:
plt.savefig("09_data_quality/demofigures/measurement_error_tau_vs_sigma.png", dpi=200)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.4.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\matplotlib\pyplot.py", line 1250, in savefig
    res = fig.savefig(*args, **kwargs)  # type: ignore[func-returns-value]
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\matplotlib\figure.py", line 3490, in savefig
    self.canvas.print_figure(fname, **kwargs)
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-package

In [ ]:
plt.savefig("09_data_quality/demo/figures/measurement_error_tau_vs_sigma.png", dpi=200)


In [ ]:
plt.close()


In [ ]:
plt.figure(figsize=(8, 5))


Figure(800x500)


In [ ]:
plt.plot(results["sigma_u"], results["beta_oracle_mean"], marker="o", label="Oracle: coef on x_true")


In [ ]:
plt.plot(results["sigma_u"], results["beta_naive_mean"], marker="o", label="Naive: coef on x_obs")


In [ ]:
plt.plot(results["sigma_u"], results["beta_cal_mean"], marker="o", label="Calibration: coef on x_hat")


In [ ]:
plt.axhline(beta, linestyle="--", label="True beta")


Line2D(True beta)


In [ ]:
plt.title("Estimated confounder effect vs measurement error (attenuation)")


Text(0.5, 1.0, 'Estimated confounder effect vs measurement error (attenuation)')


In [ ]:
plt.xlabel("Measurement error SD (sigma_u)")


Text(0.5, 0, 'Measurement error SD (sigma_u)')


In [ ]:
plt.ylabel("Estimated coefficient on confounder term")


Text(0, 0.5, 'Estimated coefficient on confounder term')


In [ ]:
plt.legend()


Legend


In [ ]:
plt.tight_layout()


In [ ]:
plt.savefig("figures/measurement_error_beta_vs_sigma.png", dpi=200)


Traceback (most recent call last):
  File "c:\Users\karra\.vscode\extensions\ms-python.python-2026.4.0-win32-x64\python_files\python_server.py", line 139, in exec_user_input
    retval = callable_(user_input, user_globals)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<string>", line 1, in <module>
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\matplotlib\pyplot.py", line 1250, in savefig
    res = fig.savefig(*args, **kwargs)  # type: ignore[func-returns-value]
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\matplotlib\figure.py", line 3490, in savefig
    self.canvas.print_figure(fname, **kwargs)
  File "C:\Users\karra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-package

In [ ]:
plt.savefig("09_data_quality/demo/figures/measurement_error_beta_vs_sigma.png", dpi=200)


In [ ]:
plt.close()


In [ ]:
sigma_u_perm = 1.0


In [ ]:
u_perm = np.random.normal(loc=0.0, scale=sigma_u_perm, size=n)


In [ ]:
x_obs_perm = x_true + u_perm


In [ ]:
ones = np.ones(n)


In [ ]:
X_obs = np.column_stack([ones, d, x_obs_perm])


In [ ]:
coef_obs, _, _, _ = np.linalg.lstsq(X_obs, y, rcond=None)


In [ ]:
tau_hat_obs = float(coef_obs[1])


In [ ]:
print("\n--- Permutation placebo setup ---")



--- Permutation placebo setup ---


In [ ]:
print("sigma_u used:", sigma_u_perm)


sigma_u used: 1.0


In [ ]:
print("Observed tau_hat (naive model):", round(tau_hat_obs, 4))


Observed tau_hat (naive model): 1.4516


In [ ]:
B = 500


In [ ]:
tau_perm = []


In [ ]:
for b in range(B):
    d_perm = np.random.permutation(d)
    X_b = np.column_stack([ones, d_perm, x_obs_perm])
    coef_b, _, _, _ = np.linalg.lstsq(X_b, y, rcond=None)
    tau_perm.append(float(coef_b[1]))



In [ ]:
tau_perm = np.array(tau_perm)


In [ ]:
p_emp = (1.0 + np.sum(np.abs(tau_perm) >= np.abs(tau_hat_obs))) / (B + 1.0)


In [ ]:
print("Empirical p-value (two-sided):", round(p_emp, 4))


Empirical p-value (two-sided): 0.002


In [ ]:
perm_df = pd.DataFrame({"tau_perm": tau_perm})


In [ ]:
perm_df = pd.DataFrame({"tau_perm": tau_perm})


In [ ]:
perm_df.to_csv("09_data_quality/demo/outputs/permutation_tau_distribution.csv", index=False)


In [ ]:
plt.figure(figsize=(8, 5))


Figure(800x500)


In [ ]:
plt.hist(tau_perm, bins=30, alpha=0.8)


(array([ 2.,  1.,  0.,  3.,  4.,  7., 10., 13., 14., 19., 16., 19., 33.,
       37., 38., 41., 28., 33., 32., 28., 32., 23., 14., 12., 13., 10.,
        9.,  5.,  3.,  1.]), array([-0.11656287, -0.1093289 , -0.10209493, -0.09486095, -0.08762698,
       -0.08039301, -0.07315904, -0.06592506, -0.05869109, -0.05145712,
       -0.04422315, -0.03698917, -0.0297552 , -0.02252123, -0.01528726,
       -0.00805328, -0.00081931,  0.00641466,  0.01364863,  0.02088261,
        0.02811658,  0.03535055,  0.04258452,  0.0498185 ,  0.05705247,
        0.06428644,  0.07152041,  0.07875439,  0.08598836,  0.09322233,
        0.1004563 ]), <BarContainer object of 30 artists>)


In [ ]:
plt.axvline(tau_hat_obs, linestyle="--", linewidth=2, label=f"Observed tau_hat = {tau_hat_obs:.3f}")


Line2D(Observed tau_hat = 1.452)


In [ ]:
plt.axvline(-tau_hat_obs, linestyle="--", linewidth=1)


Line2D(_child31)


In [ ]:
plt.title(f"Treatment permutation placebo (sigma_u={sigma_u_perm})\nEmpirical p-value = {p_emp:.3f}")


Text(0.5, 1.0, 'Treatment permutation placebo (sigma_u=1.0)\nEmpirical p-value = 0.002')


In [ ]:
plt.xlabel("Coefficient on permuted treatment")


Text(0.5, 0, 'Coefficient on permuted treatment')


In [ ]:
plt.ylabel("Count")


Text(0, 0.5, 'Count')


In [ ]:
plt.legend()


Legend


In [ ]:
plt.tight_layout()


In [ ]:
plt.savefig("09_data_quality/demo/figures/permutation_placebo_tau_hist.png", dpi=200)


In [ ]:
plt.close()


In [ ]:
print("\nDone. Outputs written to:")



Done. Outputs written to:


In [ ]:
print("\nDone. Outputs written to:")



Done. Outputs written to:


In [ ]:
print("  09_data_quality/demo/outputs/measurement_error_results.csv")


  09_data_quality/demo/outputs/measurement_error_results.csv


In [ ]:
print("  09_data_quality/demo/outputs/permutation_tau_distribution.csv")


  09_data_quality/demo/outputs/permutation_tau_distribution.csv


In [ ]:
print("  09_data_quality/demo/figures/measurement_error_tau_vs_sigma.png")


  09_data_quality/demo/figures/measurement_error_tau_vs_sigma.png


In [ ]:
print("  09_data_quality/demo/figures/measurement_error_beta_vs_sigma.png")


  09_data_quality/demo/figures/measurement_error_beta_vs_sigma.png


In [ ]:
print("  09_data_quality/demo/figures/permutation_placebo_tau_hist.png")


  09_data_quality/demo/figures/permutation_placebo_tau_hist.png
